[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/01_build_your_agent.ipynb)

# Notebook 1 - Build the Financial Agent

### Build a tool-calling agent over a bank ledger - then watch it fail *silently* on three engineered traps

Notebook **1 of 3**. Notebook 2 traces what the agent actually did; notebook 3 replaces
vibes with an automated eval suite.

**How to run:** top to bottom, one concept per cell. Your seat key goes in the
`LITELLM_API_KEY` Colab secret (🔑 sidebar - remember to switch **Notebook access**
on in each notebook) - or paste it when asked.


---
## Connect

Install, then point a client at the workshop gateway with your seat key.


In [ ]:
%pip install -q "openai>=1.40" "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "seaborn>=0.13"


In [ ]:
import time
from getpass import getpass
from openai import OpenAI, AuthenticationError

GATEWAY_URL = "https://fmlds-eval-workshop-gateway.fly.dev/v1"
AGENT_MODEL = "gpt-4o-mini"

# key resolution: Colab secret (🔑 sidebar, "Notebook access" ON) -> paste when asked.
# (The env-var path for CI/scripts lives in utils.connect, not here.)
key = ""
try:
    from google.colab import userdata
    key = (userdata.get("LITELLM_API_KEY") or "").strip()
except Exception:
    pass
key = key or getpass("Paste your workshop API key: ")

client = OpenAI(api_key=key.strip().strip("'\""), base_url=GATEWAY_URL)

try:
    t0 = time.time()
    client.chat.completions.create(model=AGENT_MODEL, messages=[{"role": "user", "content": "ping"}], max_tokens=2)
    print(f"OK  gateway reachable ({time.time() - t0:.2f}s) - environment ready")
except AuthenticationError:
    print("Key rejected - double-check it against the key sheet, then re-run this cell.")


---
## The data: a synthetic bank ledger

508 transactions, May-Oct 2026, generated for this workshop (no real financial data).
Two columns are booby-trapped - and note what's *missing*: no merchant city, no
location. That matters in failure #3.

| column | meaning | trap |
|---|---|---|
| `date` | transaction date | |
| `raw_description` | raw bank string: `STARBUCKS #1042`, `AMZN Mktp US*1A2B3C` | string fragmentation |
| `category` | 10 broad categories | |
| `amount` | **signed**: positive = income/refund, negative = expense | polarity inversion |
| `account_type` | Checking / Savings / Credit Card | |


In [ ]:
import os

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/data/transactions.csv"

try:
    df = pd.read_csv(DATA_URL)
    source = "repo"
except Exception:
    df = pd.read_csv("data/transactions.csv")
    source = "local file"
df["date"] = pd.to_datetime(df["date"])

print(f"loaded {len(df)} transactions ({source})")
df.head()


---
## Tool 1 - `fetch_data_schema`

The agent's "look before you leap" tool. A well-behaved agent calls this before writing
any pandas code.


In [ ]:
TOOLS = {}
TOOL_SPECS = []


def register_tool(name, description, parameters, fn):
    """Register a python function as a tool the model can call."""
    TOOLS[name] = fn
    TOOL_SPECS.append({
        "type": "function",
        "function": {"name": name, "description": description, "parameters": parameters},
    })


def fetch_data_schema():
    sample = df.head(5).copy()
    sample["date"] = sample["date"].dt.strftime("%Y-%m-%d")
    return {
        "columns": {c: str(t) for c, t in df.dtypes.items()},
        "n_rows": int(len(df)),
        "date_range": [str(df["date"].min().date()), str(df["date"].max().date())],
        "allowed_categories": sorted(df["category"].unique().tolist()),
        "account_types": sorted(df["account_type"].unique().tolist()),
        "sample_rows": sample.to_dict(orient="records"),
    }


register_tool(
    "fetch_data_schema",
    "Return the transaction ledger schema: column names and dtypes, row count, date range, allowed category values, account types, and 5 sample rows. Call this before writing code that references any column.",
    {"type": "object", "properties": {}, "required": []},
    fetch_data_schema,
)
print("tools:", list(TOOLS))


---
## Tool 2 - `execute_python_code`: the agent's hands

The model writes pandas code; this tool runs it. But we're executing code written by a
language model, so two cells:

**The guard** parses the model's code (without running it) and rejects imports like
`os`/`subprocess` and builtins like `open`/`eval` - the rejection goes back to the
model as a tool result, so it can rewrite and retry, like a code review.

**The executor** runs the approved snippet in a workspace holding `df` (plus
pandas/numpy/matplotlib/seaborn), keeps variables between calls, and returns what the
code printed, any traceback, and how many charts it drew. The model seeing its own
errors and retrying is the "retry bloat" of failure #3.


In [ ]:
import ast

BLOCKED = {"os", "sys", "subprocess", "requests", "urllib", "pathlib", "importlib",
           "open", "exec", "eval", "compile", "__import__", "globals", "locals", "input"}


def _guard(code_str):
    """Parse the model's code without running it; reject blocked imports and builtins."""
    try:
        tree = ast.parse(code_str)
    except SyntaxError as e:
        return f"SyntaxError: {e}"
    for node in ast.walk(tree):
        if isinstance(node, ast.Import) and any(a.name.split(".")[0] in BLOCKED for a in node.names):
            return "Blocked: imports are not allowed inside the agent sandbox."
        if isinstance(node, ast.ImportFrom) and (node.module or "").split(".")[0] in BLOCKED:
            return "Blocked: imports are not allowed inside the agent sandbox."
        if isinstance(node, ast.Name) and node.id in BLOCKED:
            return f"Blocked: use of '{node.id}' is not allowed inside the agent sandbox."
        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            return "Blocked: dunder attribute access is not allowed inside the agent sandbox."
    return None


In [ ]:
import io
import traceback
from contextlib import redirect_stderr, redirect_stdout

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

_AGENT_NS = {}
EXEC_LOG = []


def execute_python_code(code_str):
    """Run a snippet in a stateful namespace holding df, pd, np, plt, sns.
    Stdout, tracebacks and figures are captured and returned to the model."""
    _AGENT_NS.update({"df": df, "pd": pd, "np": np, "plt": plt, "sns": sns})
    blocked = _guard(code_str)
    if blocked:
        return {"ok": False, "stdout": "", "stderr": blocked, "figures": 0}
    buf_out, buf_err = io.StringIO(), io.StringIO()
    ok = True
    with redirect_stdout(buf_out), redirect_stderr(buf_err):
        try:
            exec(compile(code_str, "<agent>", "exec"), _AGENT_NS)
        except Exception:
            ok = False
            buf_err.write(traceback.format_exc())
    figures = len(plt.get_fignums())
    if figures and ok:
        plt.show()
        plt.close("all")
    entry = {"code": code_str, "ok": ok, "stderr": buf_err.getvalue()[-800:]}
    EXEC_LOG.append(entry)  # the eval suite (notebook 3) grades the agent partly on this log
    return {"ok": ok, "stdout": buf_out.getvalue()[-2000:], "stderr": entry["stderr"], "figures": figures}


register_tool(
    "execute_python_code",
    "Execute a Python snippet in a stateful namespace that already contains the pandas DataFrame `df` (the bank transaction ledger) plus pd, np, plt and sns. Printed output, tracebacks and matplotlib figures are captured and returned. Prefer several small snippets over one large one.",
    {
        "type": "object",
        "properties": {"code_str": {"type": "string", "description": "Python source to exec; the ledger DataFrame is available as df."}},
        "required": ["code_str"],
    },
    execute_python_code,
)
print("tools:", list(TOOLS))


---
## Tool 3 - `financial_rules_lookup`

Deterministic heuristics (50/30/20, emergency-fund sizing, ...). Deterministic on
purpose: advisory answers should be anchored to retrievable rules, not model vibes.


In [ ]:
FINANCIAL_RULES = [
    {"id": "50-30-20", "keywords": ["50", "30", "20", "budget", "budgeting", "allocation"], "rule": "50/30/20 rule: needs <= 50% of after-tax income, wants <= 30%, savings & debt repayment >= 20%."},
    {"id": "emergency-fund", "keywords": ["emergency", "fund", "buffer", "rainy"], "rule": "Emergency fund: hold 3-6 months of essential expenses in liquid savings; lean toward 6 if income is variable."},
    {"id": "debt-to-income", "keywords": ["debt", "dti", "ratio", "loan", "loans"], "rule": "Debt-to-income: keep total monthly debt payments below 36% of gross monthly income."},
    {"id": "housing-cost", "keywords": ["housing", "rent", "mortgage"], "rule": "Housing cost ratio: keep rent or mortgage below 30% of gross monthly income."},
    {"id": "savings-rate", "keywords": ["savings rate", "saving", "save"], "rule": "Savings rate: save at least 20% of income; 15% is a common floor for retirement-focused savers."},
    {"id": "subscription-hygiene", "keywords": ["subscription", "subscriptions", "recurring", "netflix", "spotify"], "rule": "Subscription hygiene: audit recurring charges quarterly; cancel anything unused for 60+ days."},
]


def financial_rules_lookup(query):
    """Keyword-scored lookup over deterministic finance heuristics."""
    q = query.lower()
    scored = [(sum(1 for kw in r["keywords"] if kw in q), r) for r in FINANCIAL_RULES]
    scored = [pair for pair in scored if pair[0] > 0]
    scored.sort(key=lambda pair: -pair[0])
    if not scored:
        return {"matches": [], "note": "No financial heuristic matched the query."}
    return {"matches": [r for _, r in scored[:3]]}


register_tool(
    "financial_rules_lookup",
    "Look up deterministic personal-finance heuristics (50/30/20 budgeting, emergency fund sizing, debt-to-income, housing ratio, savings rate, subscription hygiene). Returns matching rules or a no-match note.",
    {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Free-text topic, e.g. 'emergency fund' or 'budget rule'."}},
        "required": ["query"],
    },
    financial_rules_lookup,
)
print("tools:", list(TOOLS))


---
## The system prompt

We start with the kind most people actually ship - one warm, vague sentence. That is our
baseline "vibe-coded" agent. (The hardened and creative prompts arrive in notebook 3.)


In [ ]:
NAIVE_SYSTEM_PROMPT = """You are a helpful data analysis assistant. Answer the user's questions about their personal finances using the provided tools."""

print("baseline prompt: one warm, vague sentence - the kind most people actually ship")


---
## The agent loop - hand-rolled, no framework

The **router** is the model itself: we send the tool schemas with every request, it
decides which to call (`msg.tool_calls`), we dispatch and feed results back, repeat
until a final answer or the tool budget (`MAX_TOOL_CALLS = 6`) runs out.

> Routing is a capability in its own right - notebook 3 evaluates it separately from
> "was the answer right".

Each run also returns an `events` log (latency, tokens, errors per call) - notebook 2
turns it into a real tracing view.


In [ ]:
import time

MAX_TOKENS = 1500


def _chat(messages, temperature, model, tools=None):
    """One LLM call through the workshop gateway."""
    kwargs = dict(model=model, messages=messages, temperature=temperature, max_tokens=MAX_TOKENS)
    if tools:
        kwargs["tools"] = tools
    return client.chat.completions.create(**kwargs)


import json

MAX_TOOL_CALLS = 6


def dispatch_tool(tc):
    """Execute one tool call requested by the model; errors become tool output, never crashes."""
    fn = TOOLS.get(tc.function.name)
    if fn is None:
        return {"error": f"unknown tool: {tc.function.name}"}
    try:
        return fn(**json.loads(tc.function.arguments or "{}"))
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


In [ ]:
def run_agent(query, system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0, model=None, label=None, max_tool_calls=MAX_TOOL_CALLS):
    """The agent loop: the model + tool schemas do the routing; we dispatch and
    feed results back until a final answer or the tool budget runs out.

    Every LLM call and tool dispatch is appended to `events` (latency, tokens,
    errors) - notebook 2 turns that log into a real tracing view.
    """
    model = model or AGENT_MODEL
    label = label or query[:48]
    events = []
    EXEC_LOG.clear()
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": query}]
    tool_calls_used = 0
    answer = ""
    while True:
        t0 = time.time()
        resp = _chat(messages, temperature, model, tools=TOOL_SPECS)
        msg = resp.choices[0].message
        events.append({"kind": "llm", "name": model, "latency": round(time.time() - t0, 2),
                       "prompt_tokens": getattr(resp.usage, "prompt_tokens", 0) or 0,
                       "completion_tokens": getattr(resp.usage, "completion_tokens", 0) or 0})
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls or tool_calls_used >= max_tool_calls:
            answer = msg.content or ""
            break
        for tc in msg.tool_calls:
            tool_calls_used += 1
            t0 = time.time()
            out = dispatch_tool(tc)
            events.append({"kind": "tool", "name": tc.function.name, "latency": round(time.time() - t0, 2),
                           "error": out.get("ok") is False or "error" in out})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(out)[:4000]})
        if tool_calls_used >= max_tool_calls:
            messages.append({"role": "user", "content": "Tool budget exhausted - give your best final answer now."})
    return {"query": query, "label": label, "answer": answer, "messages": messages,
            "tool_calls": tool_calls_used, "executed_code": list(EXEC_LOG), "events": events,
            "model": model, "temperature": temperature, "system_prompt": system_prompt}

---
## Failure #1 - Polarity inversion (sign handling)

Ask the naive agent for October expenses. Salary and refunds are **positive**, expenses
**negative**. Watch what a "helpful" agent does with that.

Before running: what do *you* expect it to say?


In [ ]:
demo1 = run_agent("What were my total expenses in October?",
                              system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0,
                              label="naive/october-expenses")
print(demo1["answer"])


In [ ]:
print("--- executed code ---")
for c in demo1["executed_code"]:
    print(c["code"], "\n   ok:", c["ok"])

truth1 = -df.loc[(df["date"].dt.month == 10) & (df["amount"] < 0), "amount"].sum()
classic_mistake = df.loc[df["date"].dt.month == 10, "amount"].abs().sum()
print(f"\ntruth (sum of negative October amounts):   {truth1:,.2f}")
print(f"the classic mistake (sum of |amounts|):    {classic_mistake:,.2f}")
print(f"income+refund the mistake swallows:        {df.loc[(df['date'].dt.month == 10) & (df['amount'] > 0), 'amount'].sum():,.2f}")
print("\nnow look at the number the agent gave above. If it matches the second line,")
print("that is a SILENT FAILURE: fluent, confident - and it counted income as spending.")


---
## Failure #2 - Merchant string fragmentation

"How much did I spend at Starbucks?" The ledger never says `Starbucks`. It says
`STARBUCKS #1042`, `STARBUCKS COFFEE KIOSK`, `STARBUCKS STORE 08412`. Exact-equality
pandas code runs **without any error** and returns... zero.


In [ ]:
demo2 = run_agent("How much did I spend at Starbucks in total?",
                              system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0,
                              label="naive/starbucks")
print(demo2["answer"])


In [ ]:
print("--- executed code ---")
for c in demo2["executed_code"]:
    print(c["code"], "\n   ok:", c["ok"])

truth2 = -df.loc[df["raw_description"].str.contains("STARBUCKS") & (df["amount"] < 0), "amount"].sum()
print(f"\ntruth (case-insensitive substring match): {truth2:,.2f}")
print("\nthe variants hiding in raw_description:")
print(df.loc[df["raw_description"].str.contains("STARBUCKS"), "raw_description"].value_counts().to_string())
print("\nthe agent's code ran without a single error and answered $0.00.")
print("SILENT FAILURE: no crash, no warning - just a confidently wrong zero.")


---
## Failure #3 - Schema hallucination & retry bloat

"Show me spending by merchant location." There is **no location column**. A confident
model invents `merchant_city`, eats a `KeyError`, retries, and then - the truly dangerous
part - may fabricate an answer anyway.


In [ ]:
demo3 = run_agent("Show me my spending broken down by merchant location.",
                              system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0,
                              label="naive/merchant-location")
print(demo3["answer"])
print(f"\ntool calls used: {demo3['tool_calls']}/{MAX_TOOL_CALLS}")


In [ ]:
print("--- executed code ---")
for c in demo3["executed_code"]:
    print(c["code"], "\n   ok:", c["ok"], (c["stderr"][:120] if not c["ok"] else ""))

print("\nactual columns in the schema:")
print(sorted(df.columns.tolist()))
print("\nschema the agent should have consulted (fetch_data_schema):")
import json as _j
print(_j.dumps({k: v for k, v in fetch_data_schema().items() if k != "sample_rows"}, indent=2))


---
## Why "vibe-checking" fails

All three failures share one property: **the agent never crashed where the user could
see it.** Fluent, confident prose - with the bug living in the *semantics*: a swallowed
sign, an over-strict string match, an invented column. Semantics are exactly what human
eyeballs are bad at checking.

The rest of the workshop: **observe** what happened (notebook 2), **assert** what must
be true (notebook 3), **gate** on it in CI (notebook 3).

*(The code you just built also lives in the repo's canonical `utils.py`, generated from
this notebook's source. Notebook 2 imports it so you don't re-run this notebook.)*
